# 0. Radon Thesis

Matthew Thompson, fall 2026, EWU. 

## 0.1 Abstract

In this paper we investigate the basic idea of the Radon transform and its use in tomographic reconstruction. We consider the filtered back projection method of inversion, deriving it from the Fourier slice theorem. We analyze the numerical attributes of various common choices of filter kernel. We construct a filter kernel which is a signed Gaussian mixture by using machine learning. We train a nerual network which plays the role of a filter kernel. Finally, we give a brief discussion of generalized Radon transforms and the theory of Radon transforms.  

## 0.2 Contents

1. Definitions and introduction
   1. Introduction
   2. Radon Transform
   3. Filtered Back Projection
2. Techniques in Parallel beam X-Ray tomography
   1. Literature review, list of filters
   2. Numerical analysis
3. Learning of filter kernel
   1. Difference of gaussians filter window
   2. CNN filter window
   3. Application to reconstruction of probability distributions
4. Theory of Radon transform in R^2
   1. Fourier slice theorem
   2. 
5. Theory of generalized radon transforms 
   1. Helgason, Horocycles, etc
   2. Examples of applications in other domains
   3. Double fibration
   4. Distributions
6. Appendix A: Full proofs
7. Appendix B: Code snippets and figures

# 1. Definitions, basic ideas, and introduction

## 1.1 Introduction

Inverse problems involve solving $f(x) = y$ for $x$, given $f$ and $y$. We can think of this as reconstructing $x$ from $y=f(x)$, or as finding the inverse function $f^{-1}$ and applying it to $f^{-1}(y) = x$. 

Computed Tomography is such an inverse problem. 

## 1.2 Radon Transform

####  Radon transform

The Radon transform maps a function defined on $\mathbb{R}^n$ to a function defined over the set of all $n-1$ dimensional hyperplanes in $\mathbb{R}^n$. 

\begin{align*} \mathcal{R}f (r, \alpha) = \int_{\langle x, \alpha \rangle = r} f(x) dx \end{align*}




For the case of $\mathbb{R}^2$, given a function $f: \mathbb{R}^2 \to \mathbb{R}$, and the lines $\Gamma(r, a) = \{x, y \in \mathbb{R}^2 | r = x \cos(a) + y \sin(a)\}$, we define the radon transform as 

\begin{align*} \mathcal{R}f (r, a) = \int_{-\infty}^{\infty}    f(r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)       dt. \end{align*}

We say a point $(x, y)$  is incident with the line $\Gamma(r, a)$ if they "touch", that is if $(x,y) = (r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)$ for some $t$. We can capture this idea of incidence by writing the Radon transform in terms of the Dirac delta function.

\begin{align*} \mathcal{R}f (r, a) = \int \int_{\mathbb{R}^2} f(x, y) \delta (r - x \cos(a) - y \sin(a)) dx dy \end{align*}



#### Adjoint

We can also consider the dual to the radon transform, $\mathcal{R}^*$ which sends each line of the function $\mathcal{R}f(r, \alpha)$ back to a point $x \in \mathbb{R}^n$. If $g(\gamma) = \mathcal{R}f(x)$, then 

\begin{align} \mathcal{R}^* g (x) = \int_{x \in \gamma} g(\gamma) d\mu(\gamma) \end{align},

where $d\mu(\gamma)$ is the "unique probability measure on $\{\gamma | x \in \gamma\}$". (SOURCE: WIKIPEDIA)

Later on we might define more explicitly what this measure is, and use it to define the Radon transform on a generalized function $T$, by using a test function $g$ and duality: $\langle \mathcal{R}T , g \rangle = \langle T, \mathcal{R}^* g \rangle$.

In the case of $\mathbb{R}^2$, the dual radon transform is 

\begin{align*}  \end{align*}


#### Physical scenario of Tomography

Radon inversion is used in medical imaging to reconstruct images from sensor data. X-rays passing through matter experience attenuation according to known laws: $I = I_0 e^{-Al}$. Thus we have a way to physically measure the line integrals in the Radon transform. Collecting sensor data from all angles and a range of displacements gives the full radon transform data. The transform of the density function in the Radon domain is called a sinogram. 

(In real applications, fan-beam and limited angle tomography is more common. But this paper will focus on parallel beam x-ray.)

## 1.3 Filtered Back Projection

#### Issues with simple back projection

Simply applying the adjoint radon transform to a sinogram fails to invert the Radon transform. The projection data is simply smeared back accross the cartesian domain, which results in blurring and star shaped artifacts. In effect, the back projection acts as a low pass filter (WHY IS THIS? WHAT IS THE FREQUENCY CUTOFF?) and obscures details of the image which correspond to high frequencies in the Fourier domain, such as edges and fine-grained patterns.  

#### Filtered Back Projection

To avoid this, a ramp filter $|\omega|$ is introduced, which amplifies higher frequencies in the fourier domain, and subdues the low-frequency blurring.

However, introduction of this high-frequency-amplifying ramp filter makes the filtered back projection extremely sensitive to high-frequency noise, and ill-posed. 

So we introduce additional filtering $|\omega|W(\omega)$, often in the form of a window which removes the highest frequencies, while still keeping the overall ramp shape. 

Examples of filters are Ram-lak, shepp-philips, Lam, and others. We will first investigate very basic filters which are not effective.